# Biologically Inspired Neural Gadget Controller Demos
This notebook is designed as a easy einterface to call the functions that we have defined

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import scipy.stats as stats
from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import seaborn as sns

from ripser import ripser

from data_loader import (load_participants_info,
                         load_event_descriptions,
                         load_behavioral_data,
                         preprocess_data,
                         preprocess_data_stress,
                         preprocess_data_duration
                         )

Set seed for reproduciability: 42

In [ ]:
import random
import numpy as np
import torch

def set_seed(seed):
    """Sets seed for reproducibility across random, NumPy, and PyTorch."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed) 
    print(f"Seed set to {seed} for reproducibility!")

random_seed = 42
set_seed(random_seed)

# Data Preparation
Let's prepare some data first to fit our model. We are specifically using ["Locus coeruleus activity strengthens prioritized memories under arousal"](https://openneuro.org/datasets/ds002011/versions/1.0.0) dataset fror now.

In [3]:
DATASET_PATH = "data"
participants_df = load_participants_info(DATASET_PATH)
load_event_descriptions(DATASET_PATH)

df_behavior = load_behavioral_data(DATASET_PATH, "01")
for idx in range(2,11):
    sample_participant = f"0{idx}"
    df = load_behavioral_data(DATASET_PATH, sample_participant)
    df_behavior = pd.concat([df, df_behavior], ignore_index=True)

Let's preprocess our data first

In [ ]:
X, Y, X_tensor, Y_tensor, scaler_X, scaler_Y, df_clean = preprocess_data(df_behavior)

subject_id = None
X_stress = preprocess_data_stress(df_behavior, condition="AROUSING", subject_id=subject_id)
X_neurtal = preprocess_data_stress(df_behavior, condition="NEUTRAL", subject_id=subject_id)
X_long = preprocess_data_duration(df_behavior, duration="long", subject_id=subject_id)
X_short = preprocess_data_duration(df_behavior, duration="short", subject_id=subject_id)

In [5]:
X_train, X_test, Y_train, Y_test = train_test_split(X_tensor, Y_tensor, test_size=0.2, random_state=random_seed)

X_stress_train, X_stress_test = train_test_split(X_stress, test_size=0.2, random_state=random_seed)
X_neutral_train, X_neutral_test = train_test_split(X_neurtal, test_size=0.2, random_state=random_seed)
X_long_train, X_long_test = train_test_split(X_long, test_size=0.2, random_state=random_seed)
X_short_train, X_short_test = train_test_split(X_short, test_size=0.2, random_state=random_seed)

# Training

In [6]:
from train import (train_feed_forward_nn,
                   train_vanilla_lc_model,
                   train_ff_controller,
                   train_ff_uncertain_controller
                   )
from analysis.evaluation import evaluate_model

## Fully Connected Neural Network

To illustrate our idea, we want to train 2 models from math and computer science, which is our vanilla feed forward networks and an recurrent networks.

In [ ]:
model_ff = train_feed_forward_nn(X_train, Y_train,epochs=2000)
evaluate_model(model_ff, X_test, Y_test, df_clean, scaler_Y=scaler_Y)

## LCNECortex Fitter Model

Now coming to our customized LCNECortex model

In [8]:
# model_lc_vanilla = train_vanilla_lc_model(X_train, Y_train, epochs=2000)
# evaluate_model(model_lc_vanilla, X_test, Y_test, df_clean, scaler_Y=scaler_Y)

## FF Gadget Model

In [ ]:
ff_gadget = train_ff_controller(X_train, Y_train, epochs=5000, hidden_dim=124, patience=200)
evaluate_model(ff_gadget, X_test, Y_test, df_clean, scaler_Y=scaler_Y)

## Modified Uncertainty GadgetModel

We can adjust the uncertainty scale, which is deemed to be consisting of having certain randomness in the action (higher entropy, less loss, favoring more random actions)

In [ ]:
ff_uncertain_gadget = train_ff_uncertain_controller(X_tensor, Y_tensor, epochs=1000, hidden_dim=256, batch_size=256, patience=100, entropy_scale=0.2)
evaluate_model(ff_uncertain_gadget, X_test, Y_test, df_clean, scaler_Y=scaler_Y)

# Analysis of the Networks

In [ ]:
from analysis.analysis import (pca_lcne_lstm,
                               pca_feed_forward,
                               pca_lcne,
                               pca_lstm,
                               analyze_ff_gadget_activations,
                               evaluate_ff_uncertainty_gadget
)

## Feed-Forward Neural Networks

In [12]:
# pca_feed_forward(model_ff, X_tensor, df_behavior)

## LCNECortex Model

We will see that, though  under fitted with the real data, there are some structureness to the data that we can play around with since we injected mechanistic insights into it.

In [13]:
# pca_lcne(model_lc_vanilla, X_tensor, df_clean)

## Feed-Forward Neural Gadget Controller

In [ ]:
analyze_ff_gadget_activations(ff_gadget, X_tensor, df_clean)

This model elarning representation is quite unstable

## Feed-Forward Neural Gadget Uncertainty Controller

In [ ]:
evaluate_ff_uncertainty_gadget(ff_uncertain_gadget, X_tensor, Y_tensor, scaler_Y)

# Maniforlds Projections Nalaysis for Feed-Forward Neural Gadget Uncertainty Controller

In [16]:
from analysis.analysis import (extract_activations_gadget,
                               compute_mapper_graph,
                               compute_persistent_homology,
                               compute_persistent_homology_overlay,
                               compute_mapper_graph_overlay,
                               plot_manifold_projection,
                               construct_activation_graph,
                               plot_activation_graph,
                               compute_persistence_distance_df,
                               compute_persistence_hypothesis_test,
                               compare_persistence_lifetimes,
                               plot_birth_death_distributions,
                               hypothesis_test_persistence_distributions,
                               compute_autocorrelation,
                               compute_activation_entropy,
                               compute_topological_silhouette,
                               compute_betti_curves)

In [ ]:
activations_short = extract_activations_gadget(ff_uncertain_gadget, X_short_test)
activations_long = extract_activations_gadget(ff_uncertain_gadget, X_long_test)

all_activations = np.vstack([activations_short["LC"], activations_long["LC"]])
labels = np.array(["short"] * len(activations_short["LC"]) + ["long"] * len(activations_long["LC"]))

# t-SNE projection
plot_manifold_projection(all_activations, labels, method="tsne")

# UMAP projection
plot_manifold_projection(all_activations, labels, method="umap")

## Cosine Similarity Graph

We create an `cosine similarity` graph by connecting similar components in the vector space accoridng to certain similarity threshold.

In [ ]:
activations_stress = extract_activations_gadget(ff_uncertain_gadget, X_stress)
activations_neutral = extract_activations_gadget(ff_uncertain_gadget, X_neurtal)

G_stress = construct_activation_graph(activations_stress["LC"], threshold=0.96)
G_neutral = construct_activation_graph(activations_neutral["LC"], threshold=0.96)

plot_activation_graph(G_stress, title="Stressful Condition - Activation Graph")
plot_activation_graph(G_neutral, title="Neutral Condition - Activation Graph")

The stressful condition seems to be stuck in the same mode of LC activation, clustered in activation space. Remanber that LC is the controller of the rest of the model and the data here is temporal across 9 participans, so we are thinking an average evolve of `thoughts` here.

## Homology Analyiss
Conducting a persistent homology analysis, which seems to work with the `uncertainty` control model but not other models.

- Formation o higher dimension cyclic loops definately exist

In [ ]:
activations = extract_activations_gadget(ff_uncertain_gadget, X_tensor)

for key, act in activations.items():
    print(f"Analyzing {key} activations...")
    compute_persistent_homology(act, title=f"{key} - Persistent Homology")
    compute_mapper_graph(act, title=f"{key} - Mapper Graph")

## Seperated Homology Analysis

1. Loops persisting longer in `longer_duration_trials` suggest sustained neural activity patterns, possibly indicating difficulty in switching away from stress-induced cognitive states.

2. More spread out birth time of these high dimensional cyclic components

3. If using all of the data (stress/non_stress), the spread is more obvious.
    - Higher variance in stressful trials means more variation in cyclic patterns.
    - If LC-NE responses fluctuate more under stress, the model fails to stabilize activations, increasing variance. Phasic NE bursts might cause chaotic transitions rather than smooth changes.


We will start with Wasserstein distance test comparison:

- The Wasserstein distance (also known as the Earth Mover’s Distance, EMD) measures the difference between two probability distributions by quantifying the amount of "work" required to transform one distribution into another.

- After computing persistence diagrams with Ripser, we get sets of birth-death pairs representing the topological features of activations. Since these diagrams are essentially point clouds, the Wasserstein distance tells us:
    - 1️⃣ How different are the topological structures of activations between conditions?
    - 2️⃣ Do stressful activations create more persistent loops/components?
    - 3️⃣ Can we quantify significant shifts in activation topology?

In [ ]:
activations_long = extract_activations_gadget(ff_uncertain_gadget, X_long_test)
activations_short = extract_activations_gadget(ff_uncertain_gadget, X_short_test)
activations_neutral = extract_activations_gadget(ff_uncertain_gadget, X_neutral_test)
activations_stress = extract_activations_gadget(ff_uncertain_gadget, X_stress_test)

In [ ]:
activations_dict = {
    # "Long Duration": activations_long,
    # "Short Duration": activations_short,
    "Neutral": activations_neutral,
    "Stressful": activations_stress
}

df_persistence_distances = compute_persistence_distance_df(activations_dict)
df_ne = df_persistence_distances[df_persistence_distances["Key"] == "NE"]
df_ne

Empirical Baselines for Wasserstein test:
* Smaller than 0.1 → Nearly identical distributions.
* From 0.1 - 0.3 → Small but noticeable difference.
* From 0.3 - 0.6 → Moderate effect, possibly meaningful.
* Greater than 0.6 → Large divergence, likely significant.

In [ ]:
df_results = compute_persistence_hypothesis_test(activations_dict, n_permutations=1000, check_for=['NE','LC'])

Smallest p-value possible is $\frac{1}{\text{number of permutations}}$

In [ ]:
df_results

The distance for each are statsitcally different. However, we want to reason about the distribution. Let's again examine the life-time of connected components distribution directly with KS statistics and the `Kolmogorov-Smirnov (KS) test`. The two-sample K-S test is a generalization of the one-sample K-S test. Given  

$$
X_1, X_2, \dots, X_n \sim F_X \quad \text{and} \quad Y_1, Y_2, \dots, Y_m \sim F_Y
$$

and the hypotheses  

$$
H_0: F_X = F_Y \quad \text{vs} \quad H_a: F_X \neq F_Y
$$

the test statistic is given by  

$$
D_{n,m} = \sup_x \left| \hat{F}_n(x) - \hat{F}_m(x) \right|
$$

where $\hat{F}_n(x)$ and $\hat{F}_m(x)$ are the empirical distribution functions of $X_1, X_2, \dots, X_n$ and $Y_1, Y_2, \dots, Y_m$, respectively.

Intuitively, the KS test statistic measures the **maximum vertical distance** of CDF.

In [ ]:
for key in activations_long.keys():
    print(f"Analyzing {key} activations...")
    diagrams_neutral = ripser(activations_neutral[key])['dgms']
    diagrams_stressful = ripser(activations_stress[key])['dgms']

    df_lifetime_persistence_results = compare_persistence_lifetimes(diagrams_neutral, diagrams_stressful, "Neutral", "Stressful")
    print(df_lifetime_persistence_results)

Let's look at the birth and death distribution of each higher order cyclic components formed in the actiavtion space

In [ ]:
# plot_birth_death_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC', homology_dim=0)
plot_birth_death_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC', homology_dim=1)

In [ ]:
plot_birth_death_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='NE', homology_dim=1)

Let's see how different these distributions are and we will be using two non-parametric test: `Kolmogorov-Smirnov (KS) test` and `Mann-Whitney U test`

In [ ]:
hypothesis_test_persistence_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC', homology_dim=0)
hypothesis_test_persistence_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC', homology_dim=1)

In [ ]:
hypothesis_test_persistence_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='NE', homology_dim=0)
hypothesis_test_persistence_distributions(activations_dict, conditions=("Neutral", "Stressful"), activation_name='NE', homology_dim=1)

We  can look at the graphical representations

In [29]:
# for key in activations_long.keys():
#     print(f"Analyzing {key} activations...")

#     act_long = activations_long[key]
#     act_short = activations_short[key]

#     compute_persistent_homology_overlay(act_long, act_short, title=f"{key} - Persistent Homology (Long duration vs. Short duration)")
#     compute_mapper_graph_overlay(act_long, act_short, title=f"{key} - Mapper Graph (Long duration vs. Short duration)")

In [ ]:
for key in activations_neutral.keys():
    print(f"Analyzing {key} activations...")

    act_neutral = activations_neutral[key]
    act_stress = activations_stress[key]

    compute_persistent_homology_overlay(act_neutral, act_stress, title=f"{key} - Persistent Homology (Neutral vs. Stressful)")

    compute_mapper_graph_overlay(act_neutral, act_stress, title=f"{key} - Mapper Graph (Neutral vs. Stressful)")

## Topological Silhouette
Weighted average of persistence diagrams, giving an overview of prominent topological features.

- Useful for detecting global shape differences between conditions.

In [ ]:
activations_dict = {
    "Neutral": activations_neutral,
    "Stressful": activations_stress
}
compute_topological_silhouette(activations_dict, homology_dim=1, activation_name='LC')

> Stressful Condition:
- Higher persistence weight for some loops at earlier birth times.
- Suggests that loops appear earlier and persist longer.
- Possibly indicates more complex recurrent activity patterns in the LC activation landscape under stress.
- Increased persistent loops under stress → May indicate recurring states or self-reinforcing LC dynamics.

> Neutral Condition:
- More stable but lower-weight loops across different birth times.
- Suggests that loop structures are less emphasized, potentially indicating a more structured activation state.
- Flatter curve in Neutral → Suggests less fluctuating activations, possibly reflecting a more stable neuromodulatory state.

## Betti Curves
Track the number of connected components, loops, and voids as a function of filtration.

- Helps quantify when topological features appear and disappear.

In [ ]:
compute_betti_curves(activations_dict, homology_dim=1,  activation_name='NE')

> Stressful Condition:
- Higher peak Betti numbers → Stronger presence of loops.
- Longer persistence of loops at lower filtration values → More sustained recurrence patterns.
- Rapid drop-off after ~0.7 → These loops exist early but disappear quickly.

> Neutral Condition:
- Lower peak Betti numbers → Fewer loops at the same filtration scale.
- More evenly distributed across filtration values → Suggests a more fluid or adaptive topology.


> Interpretations:
- Higher early Betti numbers in stress → More self-reinforcing cyclic structures, potentially reflecting cognitive fixation or repetitive thought loo.ps
- Neutral has a more gradual decay → Supports greater cognitive flexibility, avoiding getting stuck in self-repeating cycles.
- Stress curves drop sharply → Possible instability or sudden disruptions in these loops, rather than smooth transitions.

## Temporal Auto-Correlation

- Both Neutral (blue) and Stressful (orange) conditions decay quickly, suggesting that the effect of past states diminishes over time.

- The Stressful condition seems to have **slightly higher auto-correlation** for later lags, possibly indicating **more prolonged activation states**.

In [ ]:
compute_autocorrelation(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC')

## KL Divergence Entropy Between Stress and Neutral

In [ ]:
compute_activation_entropy(activations_dict, conditions=("Neutral", "Stressful"), activation_name='LC')

In [ ]:
compute_activation_entropy(activations_dict, conditions=("Neutral", "Stressful"), activation_name='NE')

# Experiments Setting Modification

In [36]:
from analysis.analysis import (analyze_uncertainty_relationships,
                               simulate_lc_activation)

In [ ]:
uncertainty_results = analyze_uncertainty_relationships(ff_uncertain_gadget, X_tensor)

In [ ]:
pupil_high, var_high, pupil_low, var_low = simulate_lc_activation(ff_uncertain_gadget, X_test, lc_boost=0.01)
plt.figure(figsize=(8,5))
sns.histplot(var_high.squeeze(), color="red", label="High LC Uncertainty", kde=True)
sns.histplot(var_low.squeeze(), color="blue", label="Low LC Uncertainty", kde=True)
plt.xlabel("Predicted Uncertainty (Pupil Variance)")
plt.title("Effect of High vs. Low LC Activation on Uncertainty")
plt.legend()
plt.show()

In [ ]:
pupil_high, var_high, pupil_low, var_low = simulate_lc_activation(ff_uncertain_gadget, X_test, lc_boost=0.5)
plt.figure(figsize=(8,5))
sns.histplot(var_high.squeeze(), color="red", label="High LC Uncertainty", kde=True)
sns.histplot(var_low.squeeze(), color="blue", label="Low LC Uncertainty", kde=True)
plt.xlabel("Predicted Uncertainty (Pupil Variance)")
plt.title("Effect of High vs. Low LC Activation on Uncertainty")
plt.legend()
plt.show()